# Notebook 03: tối ưu, stacking, dự báo độ trôi và nhãn nhiễu

Các kết luận của notebook 01 có đổi không khi tìm siêu tham số, khi ghép mô hình, khi nhìn vào độ trôi theo thời gian, hay khi nhãn có nhiễu? Cần chạy notebook
01 trước: stacking so với CatBoost 20 seed của nó, và phép dự báo độ trôi đọc bảng chia theo thời gian của nó.

| Notebook | Nội dung | Đọc kết quả của |
|---|---|---|
| `01_benchmark_and_protocol.ipynb` | benchmark bảy họ, chia theo thời gian và các đối chứng, dòng trùng, tầng nhãn | |
| `02_transfer.ipynb` | ma trận chuyển giao giữa bốn corpus, huấn luyện gộp, khoảng tin cậy, SHAP | 01 |
| `03_robustness.ipynb` | tối ưu siêu tham số, stacking, dự báo độ trôi, nhãn nhiễu | 01 |
| `04_cnn_and_dedup.ipynb` | CNN trên ký tự, bỏ dòng trùng, bỏ tiền tố `www.`, tổng hợp cả bốn notebook | 01, 02 |

## Bước 0: Thiết lập

`setup()` (trong `p2lib/setup.py`) đọc các biến môi trường ở trên, tải bộ PhishVN từ Mendeley Data và ba corpus ngoài từ Zenodo, kiểm mã SHA-256 của từng file,
dựng cây thư mục `url_work/data/processed/` mà mã của bài đọc và ghi, rồi chuyển thư mục làm việc vào đó. Các file đã tải ở notebook trước không tải lại. Mọi kết
quả nằm trong `url_work/data/processed/p2/`, và các notebook sau đọc kết quả của notebook trước từ đó.

In [ ]:
import pandas as pd
from threadpoolctl import threadpool_limits

import p2lib
from p2lib import robustness, derived
from p2lib.checks import RES, MET_TOL, CURVE_TOL, THR_TOL, ROUND4_TOL, CNN_TOL, check, check_table, check_matrix, check_json, report

env = p2lib.setup.setup()
p2lib.setup.require(f"{RES}/p2_temporal_strict.csv", f"{RES}/p2_temporal_strict_cb_k20.csv", notebook="01")

## Bước 1: Tối ưu siêu tham số

Trên cách chia thời gian của notebook 01 (Bước 3) (một mặt nạ benign cố định, seed 0), với CatBoost, HistGB và hồi quy logistic: so mô hình mặc định với mô hình sau khi tìm siêu
tham số. Hai cách tìm có cùng ngân sách số lần thử: Grey Wolf Optimizer (6 con sói, 8 vòng, `gwo_search`) và tìm ngẫu nhiên (`random_search`). Mỗi cấu hình
được chấm bằng PR-AUC kiểm định chéo 3 phần trên tập train; cách nào điểm kiểm định chéo cao hơn thì được dùng (quy tắc chọn đặt trước), rồi mô hình được đo trên
test, trung bình 3 seed.

Bảng kỳ vọng của bài không có cột `search_seed` (chỉ có một lần tìm, seed 0), nên notebook bỏ cột đó trước khi so.

In [ ]:
robustness.run_hpo()
pd.read_csv(f"{RES}/p2_hpo.csv").drop(columns=["search_seed"]).to_csv(f"{RES}/p2_hpo.csv", index=False)
check_table("1 hyperparameter search", "p2_hpo.csv", MET_TOL)
report("1 hyperparameter search")

## Bước 2: Stacking

Các bài trước về URL lừa đảo hay dùng stacking: vài mô hình nền, rồi một mô hình thứ hai học từ xác suất ngoài mẫu của chúng (`make_stacker`). Bước này chạy công
thức đó trên hai giao thức của notebook 01 (Bước 3): random forest + XGBoost, có và không có bước chọn đặc trưng lai (`hybrid_select`, giữ 12 đặc trưng); năm tổ hợp khác
(XGBoost + LightGBM, CatBoost + HistGB, CatBoost + hồi quy logistic, CatBoost + MLP, CatBoost + HistGB + hồi quy logistic + MLP); và CatBoost + hồi quy logistic
với 20 seed để so có cặp với CatBoost 20 seed của notebook 01 (Bước 5).

In [ ]:
robustness.run_stacking(hfs=True)
check_table("2 stacking", "p2_stacking_baseline.csv", MET_TOL)
robustness.run_stacking(bases=["XGBoost+LightGBM", "CatBoost+HistGB", "CatBoost+LogReg", "CatBoost+MLP", "CatBoost+HistGB+LogReg+MLP"],
                        out=f"{RES}/p2_stacking_combos.csv", curves=f"{RES}/p2_pr_curves_stacking_combos.csv")
check_table("2 stacking", "p2_stacking_combos.csv", MET_TOL)
robustness.run_stacking(seeds=20, bases=["CatBoost+LogReg"], out=f"{RES}/p2_stacking_cblr_k20.csv", curves="")
check_table("2 stacking", "p2_stacking_cblr_k20.csv", MET_TOL)
report("2 stacking")

## Bước 3: Có dự báo được độ trôi không

Ba phép đo trên cách chia thời gian:
- `e1_decay`: chia phần train theo thời gian thành các cửa sổ, đo hiệu năng của mô hình theo khoảng cách thời gian tới tập test; độ dốc suy giảm của một họ có
  tiên đoán được họ đó tụt bao nhiêu ở notebook 01 (Bước 3) không;
- `e2_matrix`: một bộ phân loại "train hay test" giữa từng cặp khối thời gian (W1 tới W4 và TEST); AUC càng cao thì hai khối càng khác nhau;
- `e3_novelty`: độ mới từ vựng của URL test so với train có báo trước được chỗ CatBoost sai không.

Rồi FCTS: stacking trong đó mô hình thứ hai học theo chuỗi thời gian (forward chaining), so với cùng thủ tục khi thứ tự thời gian bị xáo trộn và khi chia fold
ngẫu nhiên.

Bảng suy giảm (`e1_decay`) đo PR-AUC trên từng cửa sổ thời gian nhỏ (vài trăm dòng). Xác suất của random forest là trung bình 300 cây nên có nhiều giá trị bằng
nhau; cộng song song phá các thế hoà đó theo thứ tự khác nhau giữa hai lần chạy, và trên cửa sổ nhỏ PR-AUC nhích tới cỡ 10⁻⁴. Bảng này vì vậy được so với
`THR_TOL` = 10⁻³.

In [ ]:
robustness.run_forecastability()
check_table("3 forecastability", "p2_forecastability_decay.csv", THR_TOL)
for f in ("p2_forecastability_shiftmatrix.csv", "p2_forecastability_novelty.csv"):
    check_table("3 forecastability", f, MET_TOL)
robustness.run_fcts()
check_table("3 forecastability", "p2_fcts.csv", MET_TOL)
report("3 forecastability")

## Bước 4: Kiểm toán nhãn nhiễu

Theo cách "confident learning": xác suất ngoài mẫu của từng URL (kiểm định chéo 5 phần, 5 seed, `oof_proba`); một URL bị gắn cờ khi mô hình tự tin nó thuộc lớp
kia, với ngưỡng của mỗi lớp là xác suất trung bình mà lớp đó nhận (`flags`). Tỷ lệ gắn cờ được tính riêng cho lừa đảo tầng gold, silver, bronze và cho benign.
Nếu nhãn bronze (một nguồn cộng đồng) nhiễu hơn, tỷ lệ của nó sẽ cao hơn gold.

In [ ]:
robustness.run_label_noise()
check_table("4 label noise", "label_noise_audit.csv", MET_TOL)
report("4 label noise")

## Bước 5: Các con số tính từ kết quả

Phần lớn con số của bài là trung bình và độ lệch đọc thẳng từ các bảng đã so ở trên. Một số khác được tính thêm từ các bảng đó hoặc từ chính corpus: kiểm định
có cặp, bootstrap, tương quan, và vài phép đếm lại. `derived.robustness_numbers` (trong `p2lib/derived.py`) tính lại những con số thuộc các bảng của notebook này và so
từng nhóm với `results/p2_generated.json`, file bài ghi các con số đó ra. Bốn notebook cùng phủ hết các nhóm của file đó; notebook 04 kiểm lại điều này. Hàm này còn trả bản tóm tắt ma trận khối thời gian, so với `results/p2_shiftmatrix.json`.

In [ ]:
R, shiftmatrix = derived.robustness_numbers()
for key in R:
    check_json("5 derived numbers", R[key], "results/p2_generated.json", key=key, tol=MET_TOL)
check_json("5 derived numbers", shiftmatrix, "results/p2_shiftmatrix.json", tol=MET_TOL)
report("5 derived numbers")

## Bước 6: Tổng hợp

Bảng dưới gom mọi phép kiểm của notebook này theo bước, và lưu chúng vào `data/processed/p2/notebook_checks/` để notebook 04 cộng cả bốn.

In [ ]:
p2lib.checks.summary("03")